# Initial tests

In [1]:
# import libs
import jax
import jax.numpy as jnp

# define parameters
params = {'layer1': {'W': jnp.ones((3,4)), 'b': jnp.zeros(4)}, 'layer2': { 'W': jnp.ones((4,2)), 'b': jnp.zeros(2)}}

total = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"Total parameters: {total}")

scaled = jax.tree_util.tree_map(lambda x: x * 2, params)
print(f"Scaled layer1.W: {scaled['layer1']['W']}")

Total parameters: 26
Scaled layer1.W: [[2. 2. 2. 2.]
 [2. 2. 2. 2.]
 [2. 2. 2. 2.]]


In [17]:
# Test the device
print('Devices:', jax.devices()); print('Backend:', jax.default_backend())

Devices: [CpuDevice(id=0)]
Backend: cpu


# JAX Basics for LLM Inference

This notebook covers the core JAX concepts we'll use to build nanospeed:
- JAX arrays (similar to NumPy, but with device awareness)
- `jit` for compilation
- `grad` for automatic differentiation
- `vmap` for automatic batching

These four primitives are the foundation of everything we build.

In [3]:
import jax
import jax.numpy as jnp
import numpy as np

print("JAX version:", jax.__version__)
print("Devices:", jax.devices())
print("Default backend:", jax.default_backend())

JAX version: 0.7.2
Devices: [CpuDevice(id=0)]
Default backend: cpu


In [4]:
# Functional style: create new arrays instead of modifying

x = jnp.array([1.0, 2.0, 3.0])
x_new = x.at[0].set(99.0)  # Returns a NEW array
print("Original:", x)
print("Modified:", x_new)

# This functional style is crucial for jax.jit and jax.grad
# It enables JAX's transformations to work correctly

Original: [1. 2. 3.]
Modified: [99.  2.  3.]


In [5]:
# JAX arrays are similar to NumPy but live on devices (GPU/TPU/CPU)

# NumPy array
x_np = np.array([1.0, 2.0, 3.0])
print("NumPy array:", x_np, "type:", type(x_np))

# JAX array
x_jax = jnp.array([1.0, 2.0, 3.0])
print("JAX array:", x_jax, "type:", type(x_jax))

# They feel similar, but JAX arrays are immutable!
# This will raise an error:
try:
    x_jax[0] = 99.0
except TypeError as e:
    print("Error:", e)

NumPy array: [1. 2. 3.] type: <class 'numpy.ndarray'>
JAX array: [1. 2. 3.] type: <class 'jaxlib._jax.ArrayImpl'>
Error: JAX arrays are immutable and do not support in-place item assignment. Instead of x[idx] = y, use x = x.at[idx].set(y) or another .at[] method: https://docs.jax.dev/en/latest/_autosummary/jax.numpy.ndarray.at.html


# JIT Compilation

`jax.jit` transforms a Python function into a compiled, optimized version.

Why this matters for LLM inference:
- The forward pass of a transformer is mostly matrix multiplications
- Compiled code runs 10-100× faster than Python interpretation
- For our KV cache, we'll jit-compile the attention step itself

The trade-off:
- First call is slow (compilation overhead)
- Subsequent calls are fast (cached compilation)
- JAX needs to "see" the function's structure before it can optimize

In [9]:
# A simple function without jit
def slow_multiply(x, y):
    return jnp.sin(x) * jnp.cos(y)

# Same function with jit
fast_multiply = jax.jit(slow_multiply)

# Create some data
x = jnp.array([1.0, 2.0, 3.0])
y = jnp.array([0.5, 1.0, 1.5])

# Time both
import time

# First call with jit includes compilation
start = time.time()
result_jit_1 = fast_multiply(x, y)
result_jit_1.block_until_ready()  # Wait for GPU/CPU to finish
time_jit_1 = time.time() - start
print(f"First JIT call (includes compilation): {time_jit_1*1000:.2f} ms")

# Second call with jit (cached)
start = time.time()
result_jit_2 = fast_multiply(x, y)
result_jit_2.block_until_ready()
time_jit_2 = time.time() - start
print(f"Second JIT call (cached): {time_jit_2*1000:.4f} ms")

# Without jit
start = time.time()
result_no_jit = slow_multiply(x, y)
result_no_jit.block_until_ready()
time_no_jit = time.time() - start
print(f"Without JIT: {time_no_jit*1000:.4f} ms")

print(f"\nSpeedup from JIT: {time_no_jit/time_jit_2:.1f}x")

First JIT call (includes compilation): 43.47 ms
Second JIT call (cached): 0.5724 ms
Without JIT: 66.9117 ms

Speedup from JIT: 116.9x


In [13]:
# Important: JIT specializes on shape and dtype

# This works (same shape and dtype)
fast_multiply(jnp.array([2.0, 3.0, 4.0]), jnp.array([1.0, 0.5, 0.2]))

# This triggers RECOMPILATION (different shape)
try:
    start = time.time()
    fast_multiply(jnp.array([2.0, 3.0]), jnp.array([1.0, 0.5])).block_until_ready()
    print("Time for this is ", time.time()-start)
except Exception as e:
    print("Different shape triggered recompilation")

# This triggers RECOMPILATION (different dtype)
try:
    start = time.time()
    fast_multiply(jnp.array([2, 3, 4]), jnp.array([1, 2, 3])).block_until_ready()   # int instead of float
    print("Time required for this is ", time.time()-start)
except Exception as e:
    print("Different dtype triggered recompilation")

Time for this is  0.00055694580078125
Time required for this is  0.0007002353668212891


# Automatic Batching with vmap

`jax.vmap` automatically batches a function that works on a single example
into one that works on many examples.

This is CRUCIAL for LLM inference serving:
- We don't process one user request at a time — we batch many together
- vmap lets us write code for a single token and automatically parallelize
- Without vmap, we'd need explicit loops or complex indexing

In [14]:
# Function that works on a single vector
def square(x):
    return x ** 2

# Apply to single vector
x_single = jnp.array([1.0, 2.0, 3.0])
print("Single:", square(x_single))

# Apply to batch of vectors using vmap
x_batch = jnp.array([[1.0, 2.0, 3.0],
                     [4.0, 5.0, 6.0],
                     [7.0, 8.0, 9.0]])  # shape (3, 3)

# vmap automatically adds a batch dimension
batched_square = jax.vmap(square)
result = batched_square(x_batch)
print("Batched shape:", result.shape)
print("Batched result:", result)

Single: [1. 4. 9.]
Batched shape: (3, 3)
Batched result: [[ 1.  4.  9.]
 [16. 25. 36.]
 [49. 64. 81.]]


In [15]:
# More realistic example: matrix-vector product

def matvec(W, x):
    """Single example: W is (d_out, d_in), x is (d_in,)"""
    return W @ x

# Batch version: W stays (d_out, d_in), x becomes (batch, d_in)
W = jnp.array([[1.0, 2.0],
               [3.0, 4.0],
               [5.0, 6.0]])  # (3, 2)

x_batch = jnp.array([[1.0, 1.0],
                     [2.0, 2.0],
                     [3.0, 3.0]])  # (3, 2) — batch of 3

batched_matvec = jax.vmap(matvec)
result = batched_matvec(W, x_batch)
print("Result shape:", result.shape)
print("Result:\n", result)

Result shape: (3,)
Result:
 [ 3. 14. 33.]


In [16]:
# Combining jit + vmap for maximum performance

@jax.jit
@jax.vmap
def fast_square(x):
    return x ** 2

x_batch = jnp.arange(12).reshape(4, 3).astype(jnp.float32)

# First call: compilation
import time
start = time.time()
result = fast_square(x_batch).block_until_ready()
print(f"First call (compile): {(time.time()-start)*1000:.2f} ms")

# Second call: fast
start = time.time()
result = fast_square(x_batch).block_until_ready()
print(f"Second call: {(time.time()-start)*1000:.4f} ms")

First call (compile): 26.15 ms
Second call: 0.1972 ms


## Exercises for me

In [6]:
model = {
    'conv1': {'weights': jnp.ones((3,3,16)), 'bias': jnp.zeros(16)},
    'conv2': {'weights': jnp.ones((3,3,32)), 'bias': jnp.zeros(32)},
    'fc': {'weights': jnp.ones((2048,10)), 'bias': jnp.zeros(10)}
}

### Task A: Use `tree_leaves` to get all arrays, then print:
   - Total number of arrays
   - Total number of parameters (sum of all `.size`)

In [7]:
help(jax.tree_util.tree_leaves)

Help on function tree_leaves in module jax.tree_util:

tree_leaves(tree: 'Any', is_leaf: 'Callable[[Any], bool] | None' = None) -> 'list[Leaf]'
    Alias of :func:`jax.tree.leaves`.



In [8]:
import jax
import jax.numpy as jnp
import numpy as np
import time

def slow_function(x):
    return jnp.sin(x) @ jnp.cos(x.T)

# NumPy version
x_np = np.random.randn(1000, 1000)
start = time.time()
for _ in range(10):
    _ = np.sin(x_np) @ np.cos(x_np.T)
numpy_time = time.time() - start

# JAX version (no JIT)
x_jax = jnp.array(x_np)
start = time.time()
for _ in range(10):
    _ = slow_function(x_jax).block_until_ready()
jax_no_jit = time.time() - start

# JAX version (with JIT)
fast = jax.jit(slow_function)
_ = fast(x_jax).block_until_ready()  # compile once
start = time.time()
for _ in range(10):
    _ = fast(x_jax).block_until_ready()
jax_jit = time.time() - start

print(f"NumPy:    {numpy_time:.3f}s")
print(f"JAX:      {jax_no_jit:.3f}s")
print(f"JAX+JIT:  {jax_jit:.3f}s")

NumPy:    0.958s
JAX:      0.558s
JAX+JIT:  0.336s
